In [1]:
# ==========================================
# IMPORT LIBRERIE GLOBALI
# ==========================================
from itertools import combinations
from pyvis.network import Network
from scipy.stats import fisher_exact
from sklearn.metrics import normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from statsmodels.stats.multitest import multipletests
import gseapy as gp
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import networkx as nx
import networkx.algorithms.community as nx_comm
import numpy as np
import os
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import time
import warnings

# ==========================================
# CELLA 1: IMPORT LIBRERIE E SETUP
# ==========================================

# Ignoriamo i warning sui dtypes misti per letture veloci
# warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

# Setup grafica
sns.set_theme(style="whitegrid")
print("✅ Librerie caricate con successo. Ambiente pronto.")

TARGET_GENE = 'KRAS'

COORTI = {
    "kras_pancreas": "./data_filtered/kras_pancreas",
    "kras_lung": "./data_filtered/kras_lung",
    "kras_colon": "./data_filtered/kras_colon",
    "pancreas": "./data_filtered/pancreas",
    "lung": "./data_filtered/lung",
    "colon": "./data_filtered/colon",
}

# MODIFICA: Cartella di output che legge i risultati della rete Multi-Omica (ALL)
OUTPUT_BASE_DIR = "./outputs_all"

COOCC_PARAMS = {
    "kras_pancreas": {"p_val": 0.05, "log2or": 1.0, "min_cooc": 3},
    "kras_lung":     {"p_val": 0.05, "log2or": 1.0, "min_cooc": 3},
    "kras_colon":    {"p_val": 0.05, "log2or": 1.0, "min_cooc": 3},
    "pancreas": {"p_val": 0.05, "log2or": 1.0, "min_cooc": 3},
    "lung":     {"p_val": 0.05, "log2or": 1.0, "min_cooc": 3},
    "colon":    {"p_val": 0.05, "log2or": 1.0, "min_cooc": 3}
}

ME_PARAMS = {
    "kras_pancreas": {"p_val": 0.01, "log2or": -1.0},
    "kras_lung":     {"p_val": 0.01, "log2or": -1.0},
    "kras_colon":    {"p_val": 0.01, "log2or": -1.0},
    "pancreas": {"p_val": 0.01, "log2or": -1.0},
    "lung":     {"p_val": 0.01, "log2or": -1.0},
    "colon":    {"p_val": 0.01, "log2or": -1.0}
}

✅ Librerie caricate con successo. Ambiente pronto.


In [2]:
# ==========================================
# CELLA 8: ANALISI DELLE METRICHE TOPOLOGICHE INTRACLUSTER (ESTESA)
# ==========================================

def analyze_intracluster_metrics(cohort_name, target_gene, is_full=True):
    # Determiniamo il tipo di rete per l'output
    net_type = "FULL" if is_full else f"FILTERED"
    print(f"\n" + "="*80)
    print(f"📊 METRICHE INTRACLUSTER: {cohort_name.upper()} ({net_type})")
    print("="*80)

    # 1. Definizione percorsi file
    base_dir = f"{OUTPUT_BASE_DIR}/{cohort_name}"
    stats_file = f"{base_dir}/stats/Full_Cooccurrence_Stats_{cohort_name}.tsv"
    cluster_file = f"{base_dir}/networks/Cluster_Genes_{net_type}_{cohort_name}.tsv"

    if not os.path.exists(stats_file) or not os.path.exists(cluster_file):
        print(f"[!] File mancanti per l'analisi di {cohort_name} ({net_type}).")
        return

    # 2. Ricostruzione della Rete originale (per calcolare gli archi)
    df_stats = pd.read_csv(stats_file, sep='\t')
    
    # Recupero soglie per ricostruire esattamente la rete
    p_thresh = COOCC_PARAMS[cohort_name]['p_val']
    log_thresh = COOCC_PARAMS[cohort_name]['log2or']
    min_cooc = COOCC_PARAMS[cohort_name]['min_cooc']

    valid_edges = df_stats[(df_stats['P_Value'] <= p_thresh) & 
                           (df_stats['Log2OR'] >= log_thresh) & 
                           (df_stats['Co_Occurrence_Count'] >= min_cooc)]
                           
    G_full = nx.Graph()
    for _, row in valid_edges.iterrows():
        G_full.add_edge(row['Gene_A'], row['Gene_B'], weight=int(row['Co_Occurrence_Count']))

    # Selezione tra FULL e FILTRATA
    if is_full:
        G_work = G_full
    else:
        if target_gene in G_full.nodes():
            neighbors = list(G_full.neighbors(target_gene))
            G_work = G_full.subgraph(neighbors + [target_gene]).copy()
        else:
            print(f"[!] Target {target_gene} non presente nella rete, salto calcolo.")
            return

    # 3. Caricamento dei Cluster
    df_clusters = pd.read_csv(cluster_file, sep='\t')
    cluster_ids = sorted(df_clusters['Cluster_ID'].unique())

    metrics_list = []

    # 4. Calcolo Metriche per ogni singolo cluster
    for c_id in cluster_ids:
        # Estraiamo i geni del cluster che sono effettivamente nella rete
        cluster_genes = df_clusters[df_clusters['Cluster_ID'] == c_id]['Gene'].tolist()
        valid_genes = [g for g in cluster_genes if g in G_work.nodes()]
        
        # Se un cluster ha meno di 2 nodi, le metriche topologiche non hanno senso
        if len(valid_genes) < 2:
            continue
            
        # Creiamo il sottografo contenente SOLO i geni di questo cluster
        subG = G_work.subgraph(valid_genes)
        
        n_nodes = subG.number_of_nodes()
        n_edges = subG.number_of_edges()
        
        # Calcolo Metriche Base
        density = nx.density(subG)
        avg_degree = (2 * n_edges) / n_nodes if n_nodes > 0 else 0
        avg_clustering = nx.average_clustering(subG, weight='weight')
        
        # --- NUOVE METRICHE ---
        transitivity = nx.transitivity(subG)
        n_components = nx.number_connected_components(subG)
        
        # Peso medio degli archi (Forza della co-occorrenza nel cluster)
        if n_edges > 0:
            avg_weight = sum([d.get('weight', 1) for u, v, d in subG.edges(data=True)]) / n_edges
        else:
            avg_weight = 0
            
        # Diametro (calcolato sul componente connesso più grande per evitare errori se il cluster è frammentato)
        lcc_nodes = max(nx.connected_components(subG), key=len)
        lcc = subG.subgraph(lcc_nodes)
        diameter = nx.diameter(lcc) if lcc.number_of_nodes() > 1 else 0
        
        metrics_list.append({
            'Cluster_ID': c_id,
            'N_Nodes': n_nodes,
            'N_Edges': n_edges,
            'N_Components': n_components,
            'Density': round(density, 3),
            'Avg_Degree': round(avg_degree, 2),
            'Avg_Clustering': round(avg_clustering, 3),
            'Transitivity': round(transitivity, 3),
            'Diameter_LCC': diameter,
            'Avg_Edge_Weight': round(avg_weight, 2)
        })

    # 5. Salvataggio e Stampa a video
    if metrics_list:
        df_metrics = pd.DataFrame(metrics_list)
        
        # Salva file TSV
        out_file = f"{base_dir}/intracluster/Intracluster_Metrics_{net_type}_{cohort_name}.tsv"
        os.makedirs(os.path.dirname(out_file), exist_ok=True)
        df_metrics.to_csv(out_file, sep='\t', index=False)
        
        # Stampa una tabella formattata a video
        print(df_metrics.to_string(index=False))
        print(f"\n✅ Metriche salvate in: {out_file}\n")
    else:
        print("[-] Nessun cluster valido per calcolare le metriche.\n")


# ==========================================
# ESECUZIONE AUTOMATICA PER LE 3 RETI DI OGNI CANCRO
# ==========================================
list_of_cohorts = [name for name in COORTI.keys()]

for cohort in list_of_cohorts:
    
    # CASO A: Coorti già filtrate a monte (es. 'kras_pancreas')
    if cohort.startswith('kras_'):
        analyze_intracluster_metrics(cohort, TARGET_GENE, is_full=True)
        
    # CASO B: Coorti generali (es. 'pancreas')
    else:
        analyze_intracluster_metrics(cohort, TARGET_GENE, is_full=True)
        analyze_intracluster_metrics(cohort, TARGET_GENE, is_full=False)

print("🎉 ANALISI DELLE METRICHE INTRACLUSTER ESTESE COMPLETATA!")


📊 METRICHE INTRACLUSTER: KRAS_PANCREAS (FULL)
[!] File mancanti per l'analisi di kras_pancreas (FULL).

📊 METRICHE INTRACLUSTER: KRAS_LUNG (FULL)
[!] File mancanti per l'analisi di kras_lung (FULL).

📊 METRICHE INTRACLUSTER: KRAS_COLON (FULL)
[!] File mancanti per l'analisi di kras_colon (FULL).

📊 METRICHE INTRACLUSTER: PANCREAS (FULL)
[!] File mancanti per l'analisi di pancreas (FULL).

📊 METRICHE INTRACLUSTER: PANCREAS (FILTERED)
[!] File mancanti per l'analisi di pancreas (FILTERED).

📊 METRICHE INTRACLUSTER: LUNG (FULL)
[!] File mancanti per l'analisi di lung (FULL).

📊 METRICHE INTRACLUSTER: LUNG (FILTERED)
[!] File mancanti per l'analisi di lung (FILTERED).

📊 METRICHE INTRACLUSTER: COLON (FULL)
[!] File mancanti per l'analisi di colon (FULL).

📊 METRICHE INTRACLUSTER: COLON (FILTERED)
[!] File mancanti per l'analisi di colon (FILTERED).
🎉 ANALISI DELLE METRICHE INTRACLUSTER ESTESE COMPLETATA!


In [3]:
# ==========================================
# CELLA 9: IDENTIFICAZIONE DEGLI HUB E METRICHE DI CENTRALITA' INTRACLUSTER
# ==========================================

def find_intracluster_hubs(cohort_name, target_gene, is_full=True):
    # Determiniamo il tipo di rete per l'output
    net_type = "FULL" if is_full else f"FILTERED"
    print(f"\n" + "="*80)
    print(f"🎯 RICERCA HUB INTRACLUSTER: {cohort_name.upper()} ({net_type})")
    print("="*80)

    # 1. Definizione percorsi file
    base_dir = f"{OUTPUT_BASE_DIR}/{cohort_name}"
    stats_file = f"{base_dir}/stats/Full_Cooccurrence_Stats_{cohort_name}.tsv"
    cluster_file = f"{base_dir}/networks/Cluster_Genes_{net_type}_{cohort_name}.tsv"

    if not os.path.exists(stats_file) or not os.path.exists(cluster_file):
        print(f"[!] File mancanti per l'analisi di {cohort_name} ({net_type}).")
        return

    # 2. Ricostruzione della Rete originale
    df_stats = pd.read_csv(stats_file, sep='\t')
    
    p_thresh = COOCC_PARAMS[cohort_name]['p_val']
    log_thresh = COOCC_PARAMS[cohort_name]['log2or']
    min_cooc = COOCC_PARAMS[cohort_name]['min_cooc']

    valid_edges = df_stats[(df_stats['P_Value'] <= p_thresh) & 
                           (df_stats['Log2OR'] >= log_thresh) & 
                           (df_stats['Co_Occurrence_Count'] >= min_cooc)]
                           
    G_full = nx.Graph()
    for _, row in valid_edges.iterrows():
        G_full.add_edge(row['Gene_A'], row['Gene_B']) # Uso rete unweighted per centralità puramente topologica

    # Selezione tra FULL e FILTRATA
    if is_full:
        G_work = G_full
    else:
        if target_gene in G_full.nodes():
            neighbors = list(G_full.neighbors(target_gene))
            G_work = G_full.subgraph(neighbors + [target_gene]).copy()
        else:
            print(f"[!] Target {target_gene} non presente nella rete, salto calcolo.")
            return

    # 3. Caricamento dei Cluster
    df_clusters = pd.read_csv(cluster_file, sep='\t')
    cluster_ids = sorted(df_clusters['Cluster_ID'].unique())

    all_nodes_data = []

    # 4. Calcolo Centralità per ogni singolo cluster
    for c_id in cluster_ids:
        cluster_genes = df_clusters[df_clusters['Cluster_ID'] == c_id]['Gene'].tolist()
        valid_genes = [g for g in cluster_genes if g in G_work.nodes()]
        
        # Saltiamo cluster troppo piccoli
        if len(valid_genes) < 3:
            continue
            
        # Sottografo del cluster
        subG = G_work.subgraph(valid_genes)
        
        # Calcolo centralità puramente topologiche
        deg_cent = nx.degree_centrality(subG)
        bet_cent = nx.betweenness_centrality(subG)
        clo_cent = nx.closeness_centrality(subG)
        
        for gene in valid_genes:
            all_nodes_data.append({
                'Cluster_ID': c_id,
                'Gene': gene,
                'Degree_Centrality': round(deg_cent[gene], 4),
                'Betweenness_Centrality': round(bet_cent[gene], 4),
                'Closeness_Centrality': round(clo_cent[gene], 4)
            })

    # 5. Salvataggio e Creazione Report
    if all_nodes_data:
        df_centrality = pd.DataFrame(all_nodes_data)
        
        # Ordiniamo prima per Cluster e poi per importanza (Degree decrescente)
        df_centrality = df_centrality.sort_values(by=['Cluster_ID', 'Degree_Centrality'], ascending=[True, False])
        
        # Salva file TSV completo
        out_tsv = f"{base_dir}/intracluster/Intracluster_Centrality_{net_type}_{cohort_name}.tsv"
        df_centrality.to_csv(out_tsv, sep='\t', index=False)
        print(f"✅ Tabella centralità completa salvata in: {out_tsv}")
        
        # --- CREAZIONE REPORT TESTUALE PER IL PROF ---
        out_txt = f"{base_dir}/intracluster/Report_Prof_TopHubs_{net_type}_{cohort_name}.txt"
        
        with open(out_txt, 'w', encoding='utf-8') as f:
            titolo = f"--- 👑 TOP 3 HUB GENES PER OGNI CLUSTER ({cohort_name.upper()} - {net_type}) ---"
            f.write(f"{titolo}\n")
            
            for c_id in df_centrality['Cluster_ID'].unique():
                cluster_df = df_centrality[df_centrality['Cluster_ID'] == c_id]
                
                sep = "-"*60
                head = f"\n{sep}\n🎯 CLUSTER {c_id} | Totale geni nel network: {len(cluster_df)}\n{sep}\n"
                f.write(head)
                
                # Prendiamo i top 3 per Degree Centrality
                top_3 = cluster_df.head(3)
                
                for index, row in top_3.iterrows():
                    gene_info = f"🔸 HUB: {row['Gene']}\n"
                    gene_info += f"   - Degree Centrality:      {row['Degree_Centrality']:.4f}\n"
                    gene_info += f"   - Betweenness Centrality: {row['Betweenness_Centrality']:.4f}\n"
                    f.write(gene_info)
                    
        print(f"📄 Report Top Hubs testuale salvato in: {out_txt}\n")
        print(f" Stampa a video dei Top 3 Hub per ogni cluster:")
        for c_id in df_centrality['Cluster_ID'].unique():
            cluster_df = df_centrality[df_centrality['Cluster_ID'] == c_id]
            top_3 = cluster_df.head(3)
            print(f"\n🎯 CLUSTER {c_id} | Totale geni nel network: {len(cluster_df)}")
            for index, row in top_3.iterrows():
                print(f"   🔸 HUB: {row['Gene']} - Degree Centrality: {row['Degree_Centrality']:.4f}")
    else:
        print("[-] Impossibile calcolare centralità (cluster troppo piccoli o non validi).\n")


# ==========================================
# ESECUZIONE AUTOMATICA PER LE 3 RETI DI OGNI CANCRO
# ==========================================
list_of_cohorts = [name for name in COORTI.keys()]

for cohort in list_of_cohorts:
    
    # CASO A: Coorti già filtrate a monte (es. 'kras_pancreas')
    if cohort.startswith('kras_'):
        find_intracluster_hubs(cohort, TARGET_GENE, is_full=True)
        
    # CASO B: Coorti generali (es. 'pancreas')
    else:
        find_intracluster_hubs(cohort, TARGET_GENE, is_full=True)
        find_intracluster_hubs(cohort, TARGET_GENE, is_full=False)

print("🎉 IDENTIFICAZIONE DEGLI HUB COMPLETATA!")


🎯 RICERCA HUB INTRACLUSTER: KRAS_PANCREAS (FULL)
[!] File mancanti per l'analisi di kras_pancreas (FULL).

🎯 RICERCA HUB INTRACLUSTER: KRAS_LUNG (FULL)
[!] File mancanti per l'analisi di kras_lung (FULL).

🎯 RICERCA HUB INTRACLUSTER: KRAS_COLON (FULL)
[!] File mancanti per l'analisi di kras_colon (FULL).

🎯 RICERCA HUB INTRACLUSTER: PANCREAS (FULL)
[!] File mancanti per l'analisi di pancreas (FULL).

🎯 RICERCA HUB INTRACLUSTER: PANCREAS (FILTERED)
[!] File mancanti per l'analisi di pancreas (FILTERED).

🎯 RICERCA HUB INTRACLUSTER: LUNG (FULL)
[!] File mancanti per l'analisi di lung (FULL).

🎯 RICERCA HUB INTRACLUSTER: LUNG (FILTERED)
[!] File mancanti per l'analisi di lung (FILTERED).

🎯 RICERCA HUB INTRACLUSTER: COLON (FULL)
[!] File mancanti per l'analisi di colon (FULL).

🎯 RICERCA HUB INTRACLUSTER: COLON (FILTERED)
[!] File mancanti per l'analisi di colon (FILTERED).
🎉 IDENTIFICAZIONE DEGLI HUB COMPLETATA!


In [ ]:
# ==========================================
# FILTRAGGIO HOUSEKEEPING GENES SULLE LISTE DEGLI HUB
# ==========================================
def filter_housekeeping(gene_list, hk_file="./housekeeping_genes.txt"):
    """
    Rimuove i geni housekeeping dalla lista dei geni identificati come Hub.
    Assicurati di avere il file con un gene per riga al percorso specificato.
    """
    if not os.path.exists(hk_file):
        print(f"  [-] File housekeeping non trovato in {hk_file}. Filtro HK ignorato.")
        return gene_list
    try:
        with open(hk_file, 'r') as f:
            hk_genes = set([line.strip().upper() for line in f.readlines()])
        filtered_list = [g for g in gene_list if g.upper() not in hk_genes]
        print(f"  [Filtro HK] Rimossi {len(gene_list) - len(filtered_list)} geni housekeeping.")
        return filtered_list
    except Exception as e:
        print(f"  [!] Errore durante la lettura degli housekeeping: {e}")
        return gene_list

In [5]:
# ==========================================
# FILTRAGGIO CITOBANDA (RUMORE STRUTTURALE CNA)
# ==========================================
def filter_cytoband(gene_list, cytoband_file="./data/gene_cytoband.tsv"):
    """
    Se ci sono più hub situati nella medesima citobanda (es. artefatti da grande amplificazione CNA), 
    mantiene solo il primo occorso (che è il più centrale, dato che l'input è ordinato).
    Il file atteso deve avere almeno le colonne 'Gene_Symbol' e 'Cytoband'.
    """
    if not os.path.exists(cytoband_file):
        print(f"  [-] File citobanda non trovato in {cytoband_file}. Filtro Citobanda ignorato.")
        return gene_list
    try:
        df_cyto = pd.read_csv(cytoband_file, sep='\t')
        cyto_dict = dict(zip(df_cyto['Gene_Symbol'], df_cyto['Cytoband']))
        
        seen_cytobands = set()
        filtered_list = []
        
        for gene in gene_list:
            cb = cyto_dict.get(gene, f"Unknown_{gene}")
            # Se la citobanda è sconosciuta, la consideriamo unica per non droppare il gene
            if cb not in seen_cytobands or "Unknown" in cb:
                filtered_list.append(gene)
                seen_cytobands.add(cb)
                
        print(f"  [Filtro Citobanda] Rimossi {len(gene_list) - len(filtered_list)} geni appartenenti alla stessa citobanda.")
        return filtered_list
    except Exception as e:
        print(f"  [!] Errore durante il filtraggio citobanda: {e}")
        return gene_list

In [6]:
# ==========================================
# FILTRAGGIO PARTITION COEFFICIENT (HUB MULTI-OMICI)
# ==========================================
def filter_partition_coefficient(gene_list, cohort_name, pc_threshold=0.3):
    """
    Calcola il Partition Coefficient (PC) e mantiene solo i geni in cui gli archi non
    sono totalmente sbilanciati su un solo tipo di mutazione (tipicamente dominanza CNA).
    """
    base_dir = f"{OUTPUT_BASE_DIR}/{cohort_name}/matrices"
    mut_file = f"{base_dir}/M_binary_MUT_{cohort_name}.tsv"
    cna_file = f"{base_dir}/M_binary_CNA_{cohort_name}.tsv"
    sv_file  = f"{base_dir}/M_binary_SV_{cohort_name}.tsv"
    
    if not (os.path.exists(mut_file) and os.path.exists(cna_file) and os.path.exists(sv_file)):
        print(f"  [-] Matrici singole non trovate per {cohort_name}. Filtro PC saltato.")
        return gene_list
        
    df_mut = pd.read_csv(mut_file, sep='\t', index_col=0)
    df_cna = pd.read_csv(cna_file, sep='\t', index_col=0)
    df_sv  = pd.read_csv(sv_file, sep='\t', index_col=0)
    
    filtered_list = []
    
    for gene in gene_list:
        if gene not in df_mut.columns:
            # Se il gene non è nella matrice per qualche motivo, lo manteniamo
            filtered_list.append(gene)
            continue
            
        k_mut = df_mut[gene].sum()
        k_cna = df_cna[gene].sum()
        k_sv  = df_sv[gene].sum()
        
        k_tot = k_mut + k_cna + k_sv
        if k_tot == 0:
            continue
            
        # PC formula
        pc = 1.0 - ((k_mut/k_tot)**2 + (k_cna/k_tot)**2 + (k_sv/k_tot)**2)
        
        if pc >= pc_threshold:
            filtered_list.append(gene)
            
    print(f"  [Filtro PC] Rimossi {len(gene_list) - len(filtered_list)} geni altamente polarizzati (PC < {pc_threshold}).")
    return filtered_list

In [7]:
# ==========================================
# CELLA 10: ENRICHMENT GLOBALE DEGLI HUB DELLA RETE (CON FILTRI OPZIONALI)
# ==========================================

def enrich_network_hubs(cohort_name, target_gene, is_full=True, apply_hk=False, apply_cyto=False, apply_pc=False):
    net_type = "FULL" if is_full else f"FILTERED"
    print(f"\n" + "="*80)
    print(f"🌟 ENRICHMENT DEGLI HUB: {cohort_name.upper()} ({net_type})")
    print("="*80)

    # 1. Percorsi file
    base_dir = f"{OUTPUT_BASE_DIR}/{cohort_name}/intracluster"
    enrichment_dir = f"{base_dir}/enrichment"
    os.makedirs(enrichment_dir, exist_ok=True)
    centrality_file = f"{base_dir}/Intracluster_Centrality_{net_type}_{cohort_name}.tsv"

    if not os.path.exists(centrality_file):
        print(f"[!] File centralità non trovato per {cohort_name} ({net_type}). Rete troppo piccola o Cella 9 fallita.")
        return

    df_centrality = pd.read_csv(centrality_file, sep='\t')
    
    # 2. Estrazione dei Top 3 Hub per ogni cluster
    hub_genes = []
    for c_id in df_centrality['Cluster_ID'].unique():
        cluster_df = df_centrality[df_centrality['Cluster_ID'] == c_id]
        top_hubs = cluster_df.head(3)['Gene'].tolist()
        hub_genes.extend(top_hubs)
        
    # Rimuoviamo eventuali duplicati e manteniamo l'ordine
    hub_genes = list(dict.fromkeys(hub_genes))
    print(f"🧬 Estratti inizialmente {len(hub_genes)} Hub unici dalla rete.")
    
    # --- ESECUZIONE FILTRI OPZIONALI ---
    if apply_hk:
        hub_genes = filter_housekeeping(hub_genes)
    if apply_cyto:
        hub_genes = filter_cytoband(hub_genes)
    if apply_pc:
        hub_genes = filter_partition_coefficient(hub_genes, cohort_name, pc_threshold=0.3)
    # -----------------------------------
    
    if len(hub_genes) < 2:
        print(f"[-] Operazione saltata: Solo {len(hub_genes)} Hub rimasti dopo il filtraggio. Troppo pochi per un enrichment statisticamente valido.")
        return
        
    print(f"Avvio Enrichment su {len(hub_genes)} Hub finali filtrati...")

    # 3. Analisi Enrichment
    databases = [
        'KEGG_2021_Human', 
        'GO_Biological_Process_2021', 
        'Reactome_2022'
    ]
    
    try:
        enr = gp.enrichr(gene_list=hub_genes, gene_sets=databases, organism='human', outdir=None)
        res_df = enr.results
        sig_res = res_df[res_df['P-value'] < 0.05].copy()
        time.sleep(1) # Pausa server
    except Exception as e:
        print(f"[!] Errore server Enrichr o lista geni non supportata: {e}")
        return

    # 4. Salvataggio
    if not sig_res.empty:
        # Ordiniamo per significatività
        sig_res = sig_res.sort_values(by=['P-value'], ascending=True)
        
        # Salva file TSV
        out_tsv = f"{enrichment_dir}/Enrichment_NetworkHubs_{net_type}_{cohort_name}.tsv"
        sig_res.to_csv(out_tsv, sep='\t', index=False)
        
        # --- CREAZIONE REPORT PER IL PROF ---
        out_txt = f"{enrichment_dir}/Report_NetworkHubs_Top5perDB_{net_type}_{cohort_name}.txt"
        with open(out_txt, 'w', encoding='utf-8') as f:
            titolo = f"--- 🏆 TOP 5 PATHWAY PER DATABASE — HUB GLOBALI DELLA RETE ({cohort_name.upper()} - {net_type}) ---"
            f.write(f"{titolo}\n")
            f.write(f"Geni Hub analizzati ({len(hub_genes)}): {', '.join(hub_genes)}\n")
            f.write("="*80 + "\n\n")
            
            for db_label, db_keyword in [('GO', 'GO_'), ('KEGG', 'KEGG_'), ('Reactome', 'Reactome')]:
                db_top5 = sig_res[sig_res['Gene_set'].str.contains(db_keyword, case=False)].head(5)
                if db_top5.empty:
                    continue
                f.write(f"\n  [{db_label}]\n")
                for index, row in db_top5.iterrows():
                    f.write(f"  🔸 [{row['Gene_set']}] {row['Term']}\n")
                    f.write(f"     - P-value: {row['P-value']:.2e} | Adjusted: {row['Adjusted P-value']:.2e}\n")
                    f.write(f"     - Overlap: {row['Overlap']}\n")
                    f.write(f"     - Hubs coinvolti: {row['Genes']}\n\n")
                
        print(f"✅ Analisi completata! Risultati trovati e salvati in: {out_tsv}")
        print(f" Top 5 pathway significativi per database:")
        for db_label, db_keyword in [('GO', 'GO_'), ('KEGG', 'KEGG_'), ('Reactome', 'Reactome')]:
            db_top5 = sig_res[sig_res['Gene_set'].str.contains(db_keyword, case=False)].head(5)
            if not db_top5.empty:
                print(f"  [{db_label}]")
                for index, row in db_top5.iterrows():
                    print(f"   🔸 [{row['Gene_set']}] {row['Term']}")
    else:
        print(f"[-] Nessun pathway statisticamente significativo trovato per i {len(hub_genes)} Hub di questa rete.")


# ==========================================
# ESECUZIONE AUTOMATICA PER LE 3 RETI
# ==========================================
# MODIFICA: Puoi cambiare i flag a True/False qui sotto per decidere quali filtri applicare
APPLY_HOUSEKEEPING_FILTER = True
APPLY_CYTOBAND_FILTER = True
APPLY_PARTITION_COEFFICIENT_FILTER = True

list_of_cohorts = [name for name in COORTI.keys()]

for cohort in list_of_cohorts:
    if cohort.startswith('kras_'):
        enrich_network_hubs(
            cohort, TARGET_GENE, is_full=True, 
            apply_hk=APPLY_HOUSEKEEPING_FILTER, 
            apply_cyto=APPLY_CYTOBAND_FILTER, 
            apply_pc=APPLY_PARTITION_COEFFICIENT_FILTER
        )
    else:
        enrich_network_hubs(
            cohort, TARGET_GENE, is_full=True, 
            apply_hk=APPLY_HOUSEKEEPING_FILTER, 
            apply_cyto=APPLY_CYTOBAND_FILTER, 
            apply_pc=APPLY_PARTITION_COEFFICIENT_FILTER
        )
        enrich_network_hubs(
            cohort, TARGET_GENE, is_full=False, 
            apply_hk=APPLY_HOUSEKEEPING_FILTER, 
            apply_cyto=APPLY_CYTOBAND_FILTER, 
            apply_pc=APPLY_PARTITION_COEFFICIENT_FILTER
        )

print("\n🎉 ENRICHMENT DEGLI HUB COMPLETATO!")


🌟 ENRICHMENT DEGLI HUB: KRAS_PANCREAS (FULL)
[!] File centralità non trovato per kras_pancreas (FULL). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: KRAS_LUNG (FULL)
[!] File centralità non trovato per kras_lung (FULL). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: KRAS_COLON (FULL)
[!] File centralità non trovato per kras_colon (FULL). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: PANCREAS (FULL)
[!] File centralità non trovato per pancreas (FULL). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: PANCREAS (FILTERED)
[!] File centralità non trovato per pancreas (FILTERED). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: LUNG (FULL)
[!] File centralità non trovato per lung (FULL). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: LUNG (FILTERED)
[!] File centralità non trovato per lung (FILTERED). Rete troppo piccola o Cella 9 fallita.

🌟 ENRICHMENT DEGLI HUB: COLON (FULL)
[!] File centra